Using the batchnorm and softmax derivatives, from original notebook, implement neural net again without loss.backward() but much quicker this time, and then tie it together in a full training loop and inference on val set

In [2]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [3]:
# Start with our previous MLP
words = open('names.txt', 'r').read().splitlines()
chars = sorted(list(set(''.join(words))))
s_to_i = {s: i+1 for i, s in enumerate(chars)}
s_to_i['.'] = 0
i_to_s = {i:s for s, i in s_to_i.items()}

In [4]:
block_size = 3

# create the dataset, now we have 228146 datapoints
# implement test and training splits
def build_dataset(words):
    X, Y = [], []

    for w in words:
        context = [0] * block_size
        for ch in w+'.':
            ix = s_to_i[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

import random
random.seed(42)
random.shuffle(words)
n1 = int(0.8*len(words))
n2 = int(0.9*len(words))

# create the datasets
X_train, Y_train = build_dataset(words[:n1])
X_val, Y_val = build_dataset(words[n1:n2])
X_test, Y_test = build_dataset(words[n2:])

In [5]:
# util func to compare gradients to torch version
def cmp(s, dt, t):
    exact = torch.all(dt == t.grad).item()
    approximate = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(exact):5s} | approximate {str(approximate):5s} | maxdiff: {maxdiff}')

In [15]:
# same code as last time - 
num_chars = 27
emb_dim = 10
hidden_layer_size = 64

# initialise
g = torch.Generator().manual_seed(2147483647)

# embedding matrix
C = torch.randn((num_chars, emb_dim), generator=g)

# layer 1
W1 = torch.randn((emb_dim * block_size, hidden_layer_size), generator=g) / (5/3)/(emb_dim*num_chars)**0.5
b1 = torch.randn(hidden_layer_size, generator=g) * 0.1 
# layer 2
W2 = torch.randn((hidden_layer_size, num_chars), generator=g) * 0.1
b2 = torch.randn(num_chars, generator=g) * 0.1 # 

# batch norm params - init as ones and zeros so our preactivations start off standard normal
bngain = torch.randn((1, hidden_layer_size)) * 0.1 + 1.0
bnbias = torch.randn((1, hidden_layer_size)) * 0.1

# note we are setting the params to random rather than ones or zeros - this is so we can actually inspect the gradients etc. Zeros can mask an incorrect backward pass

# running batch norm totals
running_mean = torch.zeros(hidden_layer_size)
running_var = torch.ones(hidden_layer_size)

# full params
params = [C, W1, b1, W2, b2, bngain, bnbias]
for p in params:
    p.requires_grad = True

In [18]:
# training loop
iterations = 100000
batch_size = 32
n = batch_size
momentum = 0.001

with torch.no_grad(): # BIG FLEX
    for i in range(iterations):

        # set up batches
        ix = torch.randint(0, X_train.shape[0], (batch_size, ), generator=g)
        Xb, Yb, = X_train[ix], Y_train[ix]

        # forward pass
        emb = C[Xb]
        embcat = emb.view(emb.shape[0], -1)

        hprebn = embcat @ W1 + b1

        # fast batchnorm
        bnraw = (hprebn - hprebn.mean(0, keepdim=True)) / torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5)
        hpreact = bngain * bnraw + bnbias


        # running param update
        running_mean += momentum * hprebn.mean(0) + (1-momentum) * running_mean
        running_var += momentum * hprebn.var(0, unbiased=True) + (1-momentum) * running_var


        h = torch.tanh(hpreact)

        # second layer
        logits = h @ W2 + b2

        # cross entropy
        loss = F.cross_entropy(logits, Yb)


        for p in params:
            p.grad = None

        # for t in [logits, h, hpreact, hprebn, embcat, emb]:
        #     t.retain_grad()
        # loss.backward()

        # backward pass - shorter

        # dlogits
        dlogits = F.softmax(logits, 1) # take the softmax for all logits
        dlogits[range(n), Yb] -= 1 # subtract 1 from the ones which correspond to the y index
        dlogits /= n # scale by n since we take the average

        # dh
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)

        # dhpreact
        # chain rule dloss/dhpreact = dloss/dh * dhpreact/dh
        dhpreact = dh * (1-h**2)

        # dhprebn - skip the whole batchnrom layer
        dhprebn = bngain / (n * torch.sqrt(hprebn.var(0, keepdim=True, unbiased=True) + 1e-5)) * ((n*dhpreact) - dhpreact.sum(0) - (n/(n-1)) * bnraw * (dhpreact * bnraw).sum(0))

        # dbngain
        # dloss/dbngain = dloss/dhpreact * dhpreact/dbngain
        # since hpreact = bngain * bnraw + bnbias -> dhpreact/dbgain = bnraw
        dbngain = (dhpreact * bnraw).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)

        # dembcat
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)
        demb = dembcat.view(emb.shape)

        # demb
        dC = torch.zeros_like(C)
        for k in range(Xb.shape[0]):
            for j in range(Xb.shape[1]):
                dC[Xb[k, j]] += demb[k, j]

        # collect all our gradients and do gradient descent
        grads = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]
        lr = 0.1 if i<int(iterations/2) else 0.01
        for p, grad in zip(params, grads):
            p.data += -lr * grad

        # report loss over iters
        if i % 10000 == 0:
            print(f"iteration: {i} | loss {loss.item()}")

iteration: 0 | loss 1.8924657106399536
iteration: 10000 | loss 1.901185393333435
iteration: 20000 | loss 2.1021015644073486
iteration: 30000 | loss 2.492978572845459
iteration: 40000 | loss 2.5982322692871094
iteration: 50000 | loss 1.8206727504730225
iteration: 60000 | loss 2.3849360942840576
iteration: 70000 | loss 2.7142608165740967
iteration: 80000 | loss 2.1097412109375
iteration: 90000 | loss 2.2893784046173096


In [27]:
# quickly fix means
with torch.no_grad():
    emb = C[X_train]
    embcat = emb.view(emb.shape[0], -1)

    hprebn = embcat @ W1 + b1
    running_mean = hprebn.mean(0, keepdim=True)
    running_var = hprebn.var(0, keepdim=True, unbiased=True)
running_mean

tensor([[ 1.3540,  1.9579, -0.9434, -1.2714, -2.1832,  2.2243, -0.5872,  2.1862,
         -2.9960,  1.6297,  0.8302,  1.1195,  0.4615, -1.9865, -1.5338,  1.6370,
         -1.0588,  0.2442,  1.6757, -1.2540, -2.7844,  0.8851, -1.3538, -3.1900,
         -2.8682,  1.2074, -2.6435,  1.1339, -1.5139, -1.3816, -0.3916,  1.5108,
          0.8381, -1.3550,  0.0693,  0.5522, -0.6514, -0.8052,  0.1390, -0.8555,
         -2.2410,  0.3025,  0.4761, -3.1496, -0.3589,  1.1986, -1.0381,  0.1266,
          3.3825,  1.8633, -0.3430, -0.7993,  1.0315, -1.8517,  0.7936,  1.6463,
         -0.0217, -2.0823, -0.1137, -0.3488, -0.9485, -0.3608, -0.5438,  0.9829]])

In [30]:
# inference on val batch

# Set up forward pass
emb = C[X_val]
embcat = emb.view(emb.shape[0], -1)

# linear
hprebn = embcat @ W1 + b1

# bn for inference using running
bnraw = (hprebn - running_mean)/torch.sqrt(running_var + 1e-5)
hpreact = bngain * bnraw + bnbias

# tanh
h = torch.tanh(hpreact)

# second linear layer
logits = h @ W2 + b2

# loss
loss = F.cross_entropy(logits, Y_val)
loss


tensor(2.1605, grad_fn=<NllLossBackward0>)